# Overview of Image Processing

This notebook follows along with the slides on **Image Processing** of Lecture 32: Overview of Image Processing, using the `machinevisiontoolbox` package. 

Note: Code cahnges in this notebook from the lecture are:
- For image object, .stats is a property, so use .stats instead of .stats()
- In accessing numpy array: older API used .image while current API uses .array
- In median filter, use rankfilter() instead of rank(). 

## Environment and Image Directory Setup

Detect whether the notebook is running on Google Colab (installing `matplotlib` and `machinevision-toolbox-python` if so), then import `numpy`, `matplotlib`, and the Machine Vision Toolbox (`machinevisiontoolbox`), which provides the `Image` and `Kernel` classes used throughout this notebook.

Place all required images in an `images` folder.

- **Google Colab:** Create `images` inside **My Drive**.
- **Local PC:** Create `images` in the **same directory as the notebook**.

```text
images/
├── pollen_image.tif
├── image2.tif
└── ...

In [ ]:
try:
    import google.colab
    print("Running on Colab")

    !pip install -q matplotlib machinevision-toolbox-python

    from google.colab import drive
    drive.mount("/content/drive")
    IMAGE_DIR = "/content/drive/MyDrive/images"
    COLAB = True

except ModuleNotFoundError:
    COLAB = False
    IMAGE_DIR = "images"

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

np.set_printoptions(
    linewidth=120,
    formatter={"float": lambda x: f"{0:8.4g}" if abs(x) < 1e-10 else f"{x:8.4g}"}
)

np.random.seed(0)

from machinevisiontoolbox import Image, Kernel

## 1. Digital Image Representation

### Digital Image Basics

Load a colour image into an `Image` object and explore its basic properties.

There is a critical distinction between pixel/image coordinates and array indices:

- **Image coordinates** $(u, v)$: $u$ is the horizontal axis (increasing left to right), $v$ is the vertical axis (increasing downward). The top-left pixel is $(0, 0)$.
- **Array indices** $[v, u]$: NumPy arrays are indexed as `[row, column]`. So a pixel at image coordinate $(u, v)$ corresponds to array element `[v, u]`.

For example, to access the pixel at image coordinate $(u=200, v=400)$, we index the array as `[400, 200]` (note the reversed order).

In [ ]:
street = Image.Read("street.png")

# Access a specific pixel value at image coordinate (u=200, v=400)
# Note the reversed index order [v, u]
pixel_val = street.array[400, 200]
print(pixel_val)

In [ ]:
# load color image into "Image" object
flowers = Image.Read("flowers9.png")

# basic methods of "Image" object
flowers.shape          # size of the image: (640, 426, 3)
flowers.dtype          # data type of the image: dtype('uint8')
flowers.min()          # minimum pixel value: np.uint8(0)
flowers.max()          # maximum pixel value: np.uint8(255)
flowers.stats          # pixel value statistics (mean, std, min, max) 
flowers[:, :, 2]       # extract blue plane
flowers.disp();        # show image

In [ ]:
# access numpy array from "Image" object
np_flowers = flowers.array
pix = np_flowers[400, 100, :]  # R,G,B values at pixel(100, 400)
print(pix)
# array([253,  23,   1], dtype=uint8)

plt.imshow(np_flowers[:, :, 0], cmap="gray") # get red plane

In [ ]:
# generate subimage (crop the full image) - slice the NumPy array
subimage = flowers.array[450:600, 250:400, :]  
plt.imshow(subimage)
plt.axis("off")
plt.show()

In [ ]:
# generate subimage (crop the full image) - slice the Image object
subimage = flowers[250:400, 450:600]        
subimage.disp();

#Note: Image object supports [u,v] slicing

## 2. Image Histograms and Pixel Distributions

An **image histogram** is a graphical representation of the distribution of pixel gray values within an image — it shows how many pixels exist at each intensity level.

- **Horizontal axis**: the pixel value (0–255 for a `uint8` image).
- **Vertical axis**: the pixel-value frequency, i.e. how many times a specific value occurs.
- **Bins**: an 8-bit image histogram typically uses 256 bins.

In [ ]:
street = Image.Read("street.png", mono=True)  # Load a grayscale image
h = street.hist()   # Compute the histogram (default is 256 bins for uint8 images)
h.plot()            # Plot the histogram

In [ ]:
print(f"Min: {street.min()}")
print(f"Max: {street.max()}")

# Display a single-line summary
street.stats
# Output: range=... , mean=..., sdev=...

## 3. Point Operations

Image processing operations fall into two broad categories:

- **Point operations**: the output value at any pixel $(u, v)$ depends only on the input value at the *same* coordinate: $O(u, v) = f(I(u, v))$.
- **Spatial operations**: the output value at any pixel $(u, v)$ depends on input values within a $w \times w$ **neighborhood** of that coordinate.

Note: NumPy performs modular arithmetic on `uint8` arrays (results wrap modulo 256), so converting `uint8` (0–255) to `float` (0.0–1.0) is often essential for precise mathematical modeling — e.g. `street.to("float")` and `street.to("uint8")`.

We use a `uint8` greyscale pollen image to demonstrate several point operations.

In [ ]:
I = Image.Read(f"{IMAGE_DIR}/pollen_image.tif")
plt.imshow(I.array, cmap='gray', vmin=0, vmax=255)
plt.title("Original Image")
I.hist().plot()  # plot histogram of I

### Image Negative

$$O(u, v) = 255 - I(u, v)$$

Notice that the histogram of the negative is a mirror image of the original.

In [ ]:
O = 255 - I   # arithmetic subtraction
plt.imshow(O.array, cmap='gray', vmin=0, vmax=255)
plt.title("Negative Image")
O.hist().plot()  # plot histogram of O

### Brightening

$$O(u, v) = I(u, v) + 100$$

Other arithmetic operations like scalar multiplication, division, and addition are also supported by the `Image` object. The `.apply` method takes a NumPy array and outputs a NumPy array for faster computation.

In [ ]:
O = I + 100
# (or)
O = I.apply(lambda x: x + 100)
plt.imshow(O.array, cmap='gray', vmin=0, vmax=255)
plt.title("Brightened Image");

### Thresholding

$$O(u, v) = 0 \text{ if } I(u, v) < 100, \quad O(u, v) = 255 \text{ if } I(u, v) \ge 100$$

This generates a binary image, separating pixels into 2 classes.

In [ ]:
O = (I >= 100)  # Thresholding
plt.imshow(O.array, cmap='gray', vmin=0, vmax=1)
plt.title("Thresholded Image");

### Linear Contrast Stretching

$$O(u,v) = \frac{255 \times (I(u,v) - I_{\min})}{I_{\max} - I_{\min}}$$

This stretches the pixel values to span the full 0–255 range.

In [ ]:
I_min = I.min()
I_max = I.max()

O = I.apply(lambda x: 255 * ((x - I_min) / (I_max - I_min)))  # Linear Stretching
plt.imshow(O.array, cmap='gray', vmin=0, vmax=255)
plt.title("Linear Stretched Image");

### Non-linear Gamma Stretching

$$O = 255 \left(\frac{I(u,v)}{255}\right)^{\gamma}, \quad \gamma = 7.5 \text{ (say)}$$

In [ ]:
O = I.gamma_decode(7.5)  # Gamma Stretching
plt.imshow(O.array, cmap='gray', vmin=0, vmax=255)
plt.title("Gamma Stretched Image");

### Histogram Equalization

Histogram equalization enhances image contrast by redistributing input pixel intensities (via the normalized cumulative distribution function, NCDF) so that the resulting histogram is approximately uniform (flat). It is particularly effective for images where textural details are hidden in a narrow range of gray levels.

In [ ]:
O = I.normhist()
O.disp()

In [ ]:
# Compare the original vs. equalized histograms and NCDFs
hI = I.hist()
hO = O.hist()

x = hI.x         # intensity bins
yI = hI.h        # original histogram
yO = hO.h        # equalized histogram

plt.figure()
plt.plot(x, yI, color="red", label="Original")
plt.plot(x, yO, color="green", label="Equalized")
plt.title("Histogram Comparison")
plt.xlabel("Intensity")
plt.ylabel("Frequency")
plt.legend()

ncdfI = hI.cdf  # normalized cumulative distribution
ncdfO = hO.cdf

plt.figure()
plt.plot(x, ncdfI, color="red", label="Original NCDF")
plt.plot(x, ncdfO, color="green", label="Equalized NCDF")
plt.title("NCDF Comparison")
plt.xlabel("Intensity")
plt.ylabel("Normalized Cumulative Proportion")
plt.legend()
plt.show()

## 4. Spatial Operations

Local (spatial) operations use **filters** to extract information not just at a single pixel, but from its neighborhood using convolution.

An **image filter** (or kernel, or mask) $K$ is a $w \times w$ square region/matrix centered on $(u, v)$. $w$ is generally odd with half-width $h$ such that $w = 2h + 1$. Elements of $K$ are called filter coefficients $k_{ij}$, with $i, j \in [-h, h]$. Different filters define specific operations/functions $f$ (linear or non-linear).

In [ ]:
# Read Original Image
mona = Image.Read("monalisa.png", mono=True, dtype="float")
mona.disp()

In [ ]:
# Box filter of half width size = 15 and convolve
K = Kernel.Box(h=15)
mona.convolve(K).disp();

In [ ]:
# Gaussian filter of half width size = 15, sigma=5 and convolve
K = Kernel.Gauss(sigma=5, h=15)
mona.convolve(K).disp();

## 5. Edge Filter

An **edge** is a rapid change in brightness over a space of a few pixels. The discrete first-order derivative along a horizontal profile $p$ is approximated as:

$$\frac{dp}{du} = p_u - p_{u-1}$$

In [ ]:
castle = Image.Read("castle.png", mono=True, dtype="float")
castle.disp()

profile = castle.image[360, 560:600]  # Profile p along v=360
x = np.arange(560, 600)               # actual pixel positions
plt.plot(x, profile)
plt.title("Profile at T, u = 560:600")
plt.grid()

In [ ]:
dprofile = np.diff(profile)     # difference operator
xd = np.arange(561, 600)        # shifted by 1
plt.plot(xd, dprofile)
plt.title("First-derivative at T, u = 561:600")
plt.grid()

### Symmetrical First-Order Differentiation and the Sobel Filter

$$\frac{dp}{du} = \frac{1}{2}(-p_{u-1} + p_{u+1}) \;\cong\; \text{correlation with } K = \left[-\tfrac{1}{2}, 0, \tfrac{1}{2}\right]$$

A simple and popular horizontal gradient filter is the **Sobel** filter. The vertical gradient is its transpose $K^T$.

In [ ]:
Du = Kernel.Sobel()
print(Du)

castle.convolve(Du).disp(colormap="signed")    # Horizontal Gradient
castle.convolve(Du.T).disp(colormap="signed")  # Vertical Gradient

## 6. Nonlinear Filters - Median Filter

Median filter is the primary tool for removing **impulse noise** ("salt and pepper" noise), where random pixels are set to either minimum or maximum values. The window size is defined by its half-width $h$:
- For a $3\times3$ window ($h=1$), there are 9 pixels; the median is the **rank 4** element (the 5th pixel in a 0-indexed sorted list).
- For a $5\times5$ window ($h=2$), there are 25 pixels; the median is the **rank 12** element.

In [ ]:
I = Image.Read(f"{IMAGE_DIR}/ckt_board_saltpep.tif")
I.disp()
I.rankfilter(rank=4, h=1).disp();   # rank filter

# Note: use rankfilter() instead of rank().


### Try it yourself
- Change the threshold value in the thresholding example, or the $\gamma$ value in gamma stretching, and see how the output changes.
- Try different half-widths `h` and `sigma` values for the Box and Gaussian filters and compare the amount of blur.
- Apply the median filter with a larger window (e.g. `h=2`) to the salt-and-pepper image and compare how well it removes noise vs. a Gaussian filter of similar size.